Universidad Torcuato Di Tella

Licenciatura en Tecnología Digital\
**Tecnología Digital VI: Inteligencia Artificial**

# **Taller de tuneo de arquitectura**

### **Librerías**

In [1]:
# Instalamos la librería de Weights & Biases e indicamos la clave.
!pip install wandb -qU
!export WANDB_API_KEY="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"

In [4]:
import matplotlib.pyplot as plt
import time
import torch
from torch import nn, optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, Dataset
import torch.nn.functional as F
import wandb
from sklearn.model_selection import train_test_split
import collections

### **Drive y W&B**

In [5]:
# drive.mount('/content/drive', force_remount = True)

In [6]:
# COMPLETAR: llamar a wandb.login().
wandb.login()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: ERROR Invalid API key: API key may only contain the letters A-Z, digits and underscores.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /home/ch

True

### **Transformaciones a los datos**

In [7]:
torch.manual_seed(181988)
if torch.cuda.is_available():
    torch.cuda.manual_seed(181988)

In [ ]:
train_transform = transforms.Compose([
    transforms.ToTensor(),             # Convierte las imágenes a tensores de PyTorch, en el caso de que todavía no lo sean.
    transforms.ElasticTransform(),
    transforms.Normalize((0.5,), (0.5,)),
])

# Sugerencia: se pueden agregar distintas transformaciones, como
# normalizaciones, data augmentation, etc. Todas se encuentran descriptas en la
# documentación de PyTorch.

# ¡Cuidado con el orden en que aplican las transformaciones! ¡El orden importa!

In [ ]:
val_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

### **Configuración del uso de los datos**

In [ ]:
# Hiperparámetros generales.

batch_size = 64
learning_rate = 0.01
epochs = 500
subset_train = 1200 # Sirve para entrenar con un subset para hacer más rápidamente las pruebas.
subset_val = 200    # Sirve para validar con un subset para hacer más rápidamente las pruebas.

project_name = 'td6-p18-tuneo-arquitectura'
experiment_name = 'baseline-2capas'

In [ ]:
# Validamos el tamaño de los subsets.
if subset_train % 10 != 0 or subset_val % 10 != 0 :
    raise Exception("Para que sea una estratificación correcta, el tamaño de cada subset deben ser múltiplo de 10.")
if subset_train + subset_val > 60000:
    raise Exception("Error: el subset de entrenamiento + el de validación no puede superar las 60.000 observaciones.")

### **Definición de los experimentos**

In [ ]:
if experiment_name == 'baseline-2capas':
    model = nn.Sequential(
                          nn.Linear(784, 256),
                          nn.ReLU(),
                          nn.Linear(256, 10),
                         )

In [ ]:
if experiment_name == 'deep-3capas-dropout':
    model = nn.Sequential(
                          nn.Linear(784, 512),
                          nn.ReLU(),
                          nn.Dropout(0.3),
                          nn.Linear(512, 256),
                          nn.ReLU(),
                          nn.Dropout(0.3),
                          nn.Linear(256, 10),
                         )

### **Carga de los datos**

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

In [2]:
# Descargamos y cargamos los datos.
trainset = datasets.FashionMNIST('MNIST_data/', download = True, train = True, transform = train_transform)
testset = datasets.FashionMNIST('MNIST_data/', download = True, train = False, transform = val_test_transform)

print(f'Cantidad de muestras para train: {len(trainset)}.')
print(f'Cantidad de muestras para test: {len(testset)}.')

NameError: name 'train_transform' is not defined

### **Creación de los conjuntos**

In [ ]:
def stratify_split(dataset: Dataset, train_samples: int, val_samples: int):
    train_indices = []
    val_indices = []

    NUM_CLASSES = 10
    train_samples_per_class = train_samples // NUM_CLASSES
    val_samples_per_class = val_samples // NUM_CLASSES

    train_target_counter = collections.Counter()
    val_target_counter = collections.Counter()

    for idx, data in enumerate(dataset):
        target = data[1]
        if train_target_counter[target] < train_samples_per_class:
            train_indices.append(idx)
            train_target_counter[target] += 1
        elif val_target_counter[target] < val_samples_per_class:
            val_target_counter[target] += 1
            val_indices.append(idx)

    train_dataset = Subset(dataset, train_indices)
    val_dataset = Subset(dataset, val_indices)

    return train_dataset, val_dataset

In [ ]:
trainset, valset = stratify_split(trainset, subset_train, subset_val)

print(len(trainset))
print(len(valset))

In [ ]:
trainloader = DataLoader(trainset, batch_size = batch_size, shuffle = True)
valloader = DataLoader(valset, batch_size = batch_size, shuffle = False)
testloader = DataLoader(testset, batch_size = batch_size, shuffle = False)

In [ ]:
model = model.to(device)

### **Exploración de los datos**

In [ ]:
# Examinamos una imagen.
dataiter = iter(valloader)
images, labels = next(dataiter)

print(type(images))
print(images.shape)
print(labels.shape)

In [ ]:
print(images[0].shape)
print(labels[0])
print("A continuación, veremos la imagen de una zapatilla deportiva (sneaker)).")
plt.imshow(images[0].numpy().squeeze(), cmap = 'Greys_r')

In [ ]:
print("Y otra de un pantalón.")
print(labels[24])
plt.imshow(images[24].numpy().squeeze(), cmap = 'Greys_r')

### **Un poco más de configuración**

In [ ]:
# Establecemos la función de pérdida.
criterion = nn.CrossEntropyLoss()

# Establecemos el optimizador.
optimizer = optim.SGD(model.parameters(), lr = learning_rate)

### **Y ahora sí: la ejecución**

In [ ]:
wandb.init(
    project = project_name,
    name = experiment_name,
    config = {
        'batch_size': batch_size,
        'learning_rate': learning_rate,
        'epochs': epochs,
        'subset_train': subset_train,
        'subset_val': subset_val,
    }
)

In [ ]:
train_losses, val_losses = [], []
val_accuracies = []
start = time.time()

best_epoch = 0
best_val_accuracy = 0.0

for epoch in range(epochs):

    # Inicio de la sección de entrenamiento.
    running_loss = 0
    train_correct = 0
    model.train()

    for images, labels in trainloader:
        # Aplastamos las imágenes de Fashion-MNIST a vector de largo 784.
        images = images.view(images.shape[0], -1)

        images = images.to(device)
        labels = labels.to(device)

        # Pasada de entrenamiento.
        optimizer.zero_grad()
        output = model.forward(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        _, y_preds = torch.max(output, 1)
        train_correct += (y_preds == labels).sum()

        running_loss += loss.item() * images.shape[0]
    # Fin de la sección de entrenamiento.

    # Inicio de la sección de validación.
    val_loss = 0
    val_correct = 0

    with torch.no_grad(): # Apagamos los gradientes para validación. Ahorra memoria y cómputo.
        model.eval() # Configuramos el modelo en modo evaluación.

        # Pasada de validación.
        for images, labels in valloader:
            images = images.to(device)
            labels = labels.to(device)
            images = images.view(images.shape[0], -1)
            output = model.forward(images)
            val_loss += criterion(output, labels).item() * images.shape[0]

            _, y_preds = torch.max(output, 1)

            val_correct += (y_preds == labels).sum().item()
    # Fin de la sección de validación.

    train_accuracy = 100 * train_correct / len(trainset)
    val_accuracy = 100 * val_correct / len(valset)
    val_accuracies.append(val_accuracy)

    train_losses.append(running_loss / len(trainset))
    val_losses.append(val_loss / len(valset))

    if val_accuracy > best_val_accuracy:
        best_epoch = epoch + 1
        best_val_accuracy = val_accuracy

        print(f"Guardando el modelo para la época {best_epoch}.")
        torch.save(model.state_dict(), f'{experiment_name}.pth')

    print('Epoch: {}/{}..'.format(epoch+1, epochs),
          'Training loss: {:.3f}..'.format(running_loss / len(trainset)),
          'Val loss: {:.3f}..'.format(val_loss/len(valset)),
          'Val accuracy: {:.3f}'.format(val_accuracy))

    wandb.log({
        'train_loss': running_loss / len(trainset),
        'val_loss': val_loss / len(valset),
        'train_accuracy': train_accuracy,
        'val_accuracy': val_accuracy,
    })

end = time.time()
train_time = round(end - start)

print(f'Training time: {str(train_time)} seconds.')

wandb.log({
    'train_time': train_time,
    'best_epoch': best_epoch,
    'best_val_accuracy': best_val_accuracy,
})

In [ ]:
wandb.finish()

In [ ]:
# Excepción puesta adrede para evitar, accidentalmente, ejecutar el conjunto de
# test antes de seleccionar el modelo.
raise Exception("Sólo cuando ya hayamos seleccionado el modelo 'ganador', en base al conjunto de validación, debemos ejecutar la siguiente sección, correspondiente al conjunto de evaluación.")

In [ ]:
model.load_state_dict(torch.load(f'{experiment_name}.pth'))
model.to(device)

In [ ]:
test_correct = 0

with torch.no_grad():
    model.eval()

    # Inicio de la sección de evaluación.
    # Pasada de evaluación.
    for images, labels in testloader:
        images = images.to(device)
        labels = labels.to(device)
        images = images.view(images.shape[0], -1)
        output = model.forward(images)

        _, y_preds = torch.max(output, 1)

        test_correct += (y_preds == labels).sum()
    # Fin de la sección de evaluación.

test_accuracy = 100 * test_correct / len(testset)

print(f'Accuracy en evaluación: {test_accuracy.item():.2f}.')